# Part 9: Supplementary Regime Stability Audit

This notebook runs the reproducible Part 9 Python runner in Colab. Part 9 is a supplementary audit for Part 7/8 regime stability. It does not re-estimate HMM models, does not create a new BTC allocation rule, and does not modify Part 1-8 outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

assert (PROJECT_DIR / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_DIR / 'experiments' / 'part9_regime_stability_audit' / 'run_part9_regime_stability_audit.py').exists(), 'Part 9 runner not found.'

In [ ]:
!python -m pip install -q -r experiments/part9_regime_stability_audit/requirements-part9.txt

## Path configuration

Each run directory first tries the canonical Drive path, then the downloaded zip folder structure with `*_outputs/...`.

In [ ]:
from pathlib import Path

def choose_existing(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])

INPUT_DIR = Path('data_2026/cleaned')
PART1_RUN_DIR = choose_existing('outputs/part1_btc_macro_state/colab_part1_seed42', 'outputs/part1_btc_macro_state_outputs/part1_btc_macro_state/colab_part1_seed42')
PART2_RUN_DIR = choose_existing('outputs/part2_portfolio_risk_budget/colab_part2_seed42', 'outputs/part2_portfolio_risk_budget_outputs/part2_portfolio_risk_budget/colab_part2_seed42')
PART3_RUN_DIR = choose_existing('outputs/part3_btc_state_dependence/colab_part3_seed42', 'outputs/part3_btc_state_dependence_outputs/part3_btc_state_dependence/colab_part3_seed42')
PART4_RUN_DIR = choose_existing('outputs/part4_conditional_btc_allocation/colab_part4_seed42', 'outputs/part4_conditional_btc_allocation_outputs/part4_conditional_btc_allocation/colab_part4_seed42')
PART5_RUN_DIR = choose_existing('outputs/part5_implementability_rebalancing/colab_part5_seed42', 'outputs/part5_implementability_rebalancing_outputs/part5_implementability_rebalancing/colab_part5_seed42')
PART6_RUN_DIR = choose_existing('outputs/part6_robustness_analysis/colab_part6_seed42', 'outputs/part6_robustness_analysis_outputs/part6_robustness_analysis/colab_part6_seed42')
PART7_RUN_DIR = choose_existing('outputs/part7_realtime_probabilistic_regime_robustness/colab_part7_seed42', 'outputs/part7_realtime_probabilistic_regime_robustness_outputs/part7_realtime_probabilistic_regime_robustness/colab_part7_seed42')
PART8_RUN_DIR = choose_existing('outputs/part8_uncertainty_quantification/colab_part8_seed42', 'outputs/part8_uncertainty_quantification_outputs/part8_uncertainty_quantification/colab_part8_seed42')
OUTPUT_DIR = Path('outputs/part9_regime_stability_audit')
RUN_ID = 'colab_part9_seed42'
SEED = 42

required_paths = {
    'INPUT_DIR': INPUT_DIR,
    'PART1_RUN_DIR': PART1_RUN_DIR,
    'PART2_RUN_DIR': PART2_RUN_DIR,
    'PART3_RUN_DIR': PART3_RUN_DIR,
    'PART4_RUN_DIR': PART4_RUN_DIR,
    'PART5_RUN_DIR': PART5_RUN_DIR,
    'PART6_RUN_DIR': PART6_RUN_DIR,
    'PART7_RUN_DIR': PART7_RUN_DIR,
    'PART8_RUN_DIR': PART8_RUN_DIR,
}
for name, path in required_paths.items():
    print(f'{name:15s}', path.exists(), path)
    assert path.exists(), f'{name} not found. Check the path configuration cell.'

In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    'experiments/part9_regime_stability_audit/run_part9_regime_stability_audit.py',
    '--input-dir', str(INPUT_DIR),
    '--part1-run-dir', str(PART1_RUN_DIR),
    '--part2-run-dir', str(PART2_RUN_DIR),
    '--part3-run-dir', str(PART3_RUN_DIR),
    '--part4-run-dir', str(PART4_RUN_DIR),
    '--part5-run-dir', str(PART5_RUN_DIR),
    '--part6-run-dir', str(PART6_RUN_DIR),
    '--part7-run-dir', str(PART7_RUN_DIR),
    '--part8-run-dir', str(PART8_RUN_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--run-id', RUN_ID,
    '--seed', str(SEED),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
import pandas as pd

RUN_DIR = OUTPUT_DIR / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
print('Run directory:', RUN_DIR)
print('Results:', sorted(p.name for p in RESULTS.iterdir()))
print('Figures:', sorted(p.name for p in FIGURES.iterdir()))

In [ ]:
display(pd.read_csv(RESULTS / 'regime_stability_overview.csv'))
display(pd.read_csv(RESULTS / 'state_confusion_matrix_counts.csv'))
display(pd.read_csv(RESULTS / 'state_stability_by_part1_state.csv'))
display(pd.read_csv(RESULTS / 'state_instability_episodes.csv'))

In [ ]:
display(pd.read_csv(RESULTS / 'rule_signal_instability_impact.csv').head(16))
display(pd.read_csv(RESULTS / 'ensemble_stability_summary.csv'))
display(pd.read_csv(RESULTS / 'ensemble_rule_sensitivity_summary.csv'))
display(pd.read_csv(RESULTS / 'regime_stability_decision_matrix.csv'))

In [ ]:
from IPython.display import Image, display

for name in [
    'state_confusion_heatmap.png',
    'state_agreement_timeline.png',
    'btc_weight_delta_by_state_pair.png',
    'ensemble_agreement_summary.png',
    'rule_signal_sensitivity.png',
    'instability_episode_lengths.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
import shutil

zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Created zip:', zip_path)